# Full Bayesian Operator Inference: Cubic Heat Equation

**Workflow:**
1. Generate training data (multiple ICs) and fit POD basis
2. Grid search for prior operator
3. Fit GP hyperparameters
4. Run Bayesian inference (SVI/MCMC)
5. Evaluate and visualize results (including test IC)

## 1. Setup and Configuration

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import jax
import jax.numpy as jnp
import opinf
import matplotlib.pyplot as plt
import numpyro
import numpyro.distributions as dist

from core import (
    JaxCompatibleModel,
    grid_search_prior_operator,
    fit_gp_hyperparameters_mle,
    compute_gp_derivatives,
    run_svi,
    run_mcmc,
    generate_rom_predictions,
)
from config import (
    FullOrderModel, Basis, ReducedOrderModel,
    input_func_factory, input_parameters, test_parameters,
    time_domain
)
from heat_plotter import HeatPlotter

numpyro.set_platform('cpu')
numpyro.set_host_device_count(4)

# Reproducibility
np.random.seed(42)
rng_key = jax.random.PRNGKey(42)

In [ ]:
# === EXPERIMENT CONFIGURATION ===
OPERATORS = "cAHBN"       # OpInf operator structure (with inputs)
NUM_MODES = 10            # POD modes to retain
NUM_TRAIN_ICS = 3         # Number of training ICs to use

# Inference settings
RUN_SVI = True
RUN_MCMC = False  # More expensive, optional

# Hyperparameters
GAMMA = 0.1      # Operator prior variance
GAMMA2 = 0.1     # ODE constraint stiffness

## 2. Generate Training Data

In [ ]:
# Select training and test parameters
train_params = input_parameters[:NUM_TRAIN_ICS]
test_params = test_parameters

print(f"Training parameters: {train_params}")
print(f"Test parameters: {test_params}")

In [ ]:
# Generate training trajectories
all_snapshots = []
all_inputs = []

for params in train_params:
    fom = FullOrderModel(params)
    fom.solve(time_domain, fom.initial_conditions)
    input_func = input_func_factory(params)
    
    all_snapshots.append(fom.snapshots)
    all_inputs.append(input_func(time_domain))

# Stack all training data
snapshots_train = np.hstack(all_snapshots)
inputs_train = np.hstack(all_inputs)
time_train = np.tile(time_domain, NUM_TRAIN_ICS)

print(f"Training snapshots: {snapshots_train.shape}")
print(f"Training inputs: {inputs_train.shape}")

In [ ]:
# Generate test trajectory
fom_test = FullOrderModel(test_params)
fom_test.solve(time_domain, fom_test.initial_conditions)
test_input_func = input_func_factory(test_params)

snapshots_test = fom_test.snapshots
inputs_test = test_input_func(time_domain)

print(f"Test snapshots: {snapshots_test.shape}")

In [ ]:
# Fit POD basis (with lifting: q, q^2)
basis = Basis(num_vectors=NUM_MODES)
basis.fit(snapshots_train)

snapshots_compressed = basis.compress(snapshots_train)
snapshots_test_compressed = basis.compress(snapshots_test)

print(f"Compressed shape: {snapshots_compressed.shape}")
print(f"Cumulative energy: {basis.cumulative_energy(NUM_MODES):.4%}")

In [ ]:
# Subsample for training
SAMPLE_RATE = 2
n_time = len(time_domain)

# Sample each trajectory
time_sampled = time_domain[::SAMPLE_RATE]
indices = []
for i in range(NUM_TRAIN_ICS):
    indices.extend(list(range(i*n_time, (i+1)*n_time, SAMPLE_RATE)))

snapshots_sampled = snapshots_train[:, indices]
snapshots_comp_sampled = snapshots_compressed[:, indices]
inputs_sampled = inputs_train[:, indices]

# For grid search, use first trajectory
first_traj_idx = list(range(0, n_time, SAMPLE_RATE))
snapshots_first = snapshots_train[:, first_traj_idx]
snapshots_comp_first = snapshots_compressed[:, first_traj_idx]
inputs_first = inputs_train[:, first_traj_idx]

print(f"Training samples per traj: {len(time_sampled)}")
print(f"Total training samples: {len(indices)}")

## 3. Grid Search for Prior Operator

In [ ]:
# Find best deterministic operator via regularization grid search
# Use first training trajectory for initial search
first_input_func = input_func_factory(train_params[0])

result = grid_search_prior_operator(
    basis=basis,
    time_domain_sampled=time_sampled,
    snapshots_sampled=snapshots_first,
    snapshots_compressed=snapshots_comp_first,
    operators=OPERATORS,
    inputs=inputs_first.T,
    input_func=first_input_func,
    verbose=True
)

prior_operator = result.operator
rom = result.rom
print(f"\nPrior operator shape: {prior_operator.shape}")

## 4. Fit GP Hyperparameters

In [ ]:
# Fit GP for each mode via MLE (on first trajectory)
Ls, Vs, Ns, gp_models = fit_gp_hyperparameters_mle(
    time_domain=time_sampled,
    snapshots=snapshots_comp_first,
    verbose=True
)

## 5. Bayesian Inference

In [ ]:
# Define Bayesian model (with inputs)
def bayesian_opinf_model(time, gamma=1e-1, gamma2=1e2, normalization=1e-6):
    """Bayesian operator inference model with GP-derived ODE constraints."""
    num_time_steps = time.shape[0]
    
    # Sample operator from prior
    O = numpyro.sample(
        "O",
        dist.Normal(loc=prior_operator, scale=gamma * jnp.ones_like(prior_operator))
    )
    
    # Latent states (GP means)
    Xs = []
    for i in range(NUM_MODES):
        mean_pred, _ = gp_models[i].predict(time[:, None], return_std=True)
        X = numpyro.deterministic(f"X{i}", jnp.array(mean_pred))
        Xs.append(X)
    Xs = jnp.array(Xs)
    
    # Compute operator dynamics (with inputs)
    inputs_eval = jnp.array(first_input_func(time)).T
    f_Xi = rom.model._assemble_data_matrix(Xs, inputs=inputs_eval) @ O.T
    
    # GP derivatives
    mu_z, cov_z = compute_gp_derivatives(
        Ls, Vs, time_sampled, time, snapshots_comp_first
    )
    
    # ODE constraints
    for i in range(NUM_MODES):
        constraint_cov = cov_z[i] + gamma2 * jnp.eye(num_time_steps)
        numpyro.sample(
            f'ode_constraint{i}',
            dist.MultivariateNormal(f_Xi.T[i], constraint_cov),
            obs=mu_z[i]
        )

In [ ]:
# Run SVI
if RUN_SVI:
    time_eval = time_sampled
    svi_result = run_svi(
        model=bayesian_opinf_model,
        rng_key=rng_key,
        time_eval=time_eval,
        gamma=GAMMA,
        gamma2=GAMMA2,
        num_steps=30000,
        learning_rate=1e-4,
        verbose=True
    )
    samples = svi_result.samples
    
    # Plot loss
    plt.figure(figsize=(10, 4))
    plt.plot(svi_result.losses)
    plt.xlabel('Iteration')
    plt.ylabel('ELBO Loss')
    plt.title('SVI Convergence')
    plt.show()

In [ ]:
# Run MCMC (optional, more expensive)
if RUN_MCMC:
    # Initialize from SVI if available
    init_values = {"O": samples['O'].mean(axis=0)} if RUN_SVI else {}
    
    mcmc_result = run_mcmc(
        model=bayesian_opinf_model,
        rng_key=jax.random.PRNGKey(1),
        time_eval=time_eval,
        init_values=init_values,
        gamma=GAMMA,
        gamma2=GAMMA2,
        num_warmup=500,
        num_samples=500,
        num_chains=2,
        verbose=True
    )
    samples = mcmc_result.samples

## 6. Results and Visualization

In [ ]:
# Generate ROM predictions from posterior (training trajectory)
Os, Xs, rom_solves = generate_rom_predictions(
    samples=samples,
    rom=rom,
    snapshots_compressed=snapshots_comp_first,
    time_eval=time_eval,
    num_modes=NUM_MODES,
    num_pulls=200,
    input_func=first_input_func,
    data_scaler=None
)

print(f"Collected {len(Os)} operator samples")
print(f"Stable ROM solves: {len(rom_solves)}")

In [ ]:
# Operator posterior summary
O_mean = Os.mean(axis=0)
O_std = Os.std(axis=0)

print("Prior operator (first row):")
print(prior_operator[0, :5])
print("\nPosterior mean (first row):")
print(O_mean[0, :5])
print("\nPosterior std (first row):")
print(O_std[0, :5])

In [ ]:
# State reconstruction (training)
if len(rom_solves) > 0:
    rom_mean = rom_solves.mean(axis=0)
    rom_std = rom_solves.std(axis=0)
    
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for i, ax in enumerate(axes.flat):
        if i >= NUM_MODES:
            ax.axis('off')
            continue
        ax.plot(time_eval, snapshots_comp_first[i], 'k-', label='Truth', lw=2)
        ax.plot(time_eval, rom_mean[i], 'b--', label='ROM Mean', lw=1.5)
        ax.fill_between(
            time_eval,
            rom_mean[i] - 2*rom_std[i],
            rom_mean[i] + 2*rom_std[i],
            alpha=0.3, color='blue', label='95% CI'
        )
        ax.set_xlabel('Time')
        ax.set_ylabel(f'Mode {i}')
        if i == 0:
            ax.legend(fontsize=8)
    plt.suptitle('Training Trajectory Reconstruction')
    plt.tight_layout()
    plt.show()
else:
    print("No stable ROM solutions found")

In [ ]:
# Compute training prediction errors
if len(rom_solves) > 0:
    errors = []
    for sol in rom_solves:
        err = np.linalg.norm(sol - snapshots_comp_first) / np.linalg.norm(snapshots_comp_first)
        errors.append(err)
    
    print("Training Trajectory Errors:")
    print(f"  Mean relative error: {np.mean(errors):.4%}")
    print(f"  Std relative error: {np.std(errors):.4%}")

## 7. Test Trajectory Generalization

In [ ]:
# Predict on test trajectory
time_test = time_domain[::SAMPLE_RATE]
test_compressed = snapshots_test_compressed[:, ::SAMPLE_RATE]

test_rom_solves = []
for O in Os:
    rom.model._extract_operators(np.array(O))
    try:
        rom.model.predict(
            state0=test_compressed[:, 0],
            t=time_test,
            input_func=test_input_func
        )
        if rom.model.predict_result_.y.shape[1] >= time_test.size:
            test_rom_solves.append(rom.model.predict_result_.y)
    except:
        pass

print(f"Test trajectory stable solves: {len(test_rom_solves)}")

In [ ]:
# Test trajectory reconstruction
if len(test_rom_solves) > 0:
    test_rom_mean = np.array(test_rom_solves).mean(axis=0)
    test_rom_std = np.array(test_rom_solves).std(axis=0)
    
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for i, ax in enumerate(axes.flat):
        if i >= NUM_MODES:
            ax.axis('off')
            continue
        ax.plot(time_test, test_compressed[i], 'k-', label='Truth', lw=2)
        ax.plot(time_test, test_rom_mean[i], 'r--', label='ROM Mean', lw=1.5)
        ax.fill_between(
            time_test,
            test_rom_mean[i] - 2*test_rom_std[i],
            test_rom_mean[i] + 2*test_rom_std[i],
            alpha=0.3, color='red', label='95% CI'
        )
        ax.set_xlabel('Time')
        ax.set_ylabel(f'Mode {i}')
        if i == 0:
            ax.legend(fontsize=8)
    plt.suptitle(f'Test Trajectory (params={test_params})')
    plt.tight_layout()
    plt.show()
    
    # Test errors
    test_errors = []
    for sol in test_rom_solves:
        err = np.linalg.norm(sol - test_compressed) / np.linalg.norm(test_compressed)
        test_errors.append(err)
    
    print("\nTest Trajectory Errors:")
    print(f"  Mean relative error: {np.mean(test_errors):.4%}")
    print(f"  Std relative error: {np.std(test_errors):.4%}")

In [ ]:
# Summary
print("=" * 50)
print("EXPERIMENT SUMMARY: Cubic Heat Equation")
print("=" * 50)
print(f"Operators: {OPERATORS}")
print(f"Modes: {NUM_MODES}")
print(f"Training ICs: {NUM_TRAIN_ICS}")
print(f"Prior regularization: {result.best_reg:.1e}")
print(f"Prior error: {result.best_error:.4%}")
print(f"Gamma (operator): {GAMMA}")
print(f"Gamma2 (ODE): {GAMMA2}")
if len(rom_solves) > 0:
    print(f"\nTraining:")
    print(f"  Posterior mean error: {np.mean(errors):.4%}")
    print(f"  Stable solutions: {len(rom_solves)}/{len(Os)}")
if len(test_rom_solves) > 0:
    print(f"\nTest:")
    print(f"  Posterior mean error: {np.mean(test_errors):.4%}")
    print(f"  Stable solutions: {len(test_rom_solves)}/{len(Os)}")